# ebms-sim: Physics-Based Simulator & Synthetic Data Generator for E-Rickshaw BMS

This notebook demonstrates the core capabilities of **ebms-sim**:
1. **Full 1-Second Trip Simulation**: Longitudinal vehicle dynamics, equivalent circuit battery model, and lumped pack thermal model.
2. **Cruising Speed Optimization**: Wh/km vs speed U-curve and dynamic programming segment scheduling.
3. **Smart Charging**: Bell-shaped thermal current derating and CC-CV taper simulation.
4. **Synthetic Data Generation & Machine Learning**: Generating ML-ready datasets and training XGBoost regressors with `GroupKFold` on `route_id`.

In [ ]:
import os
import tempfile
os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ebms_sim.config import PRESETS, VehicleParams, BatteryParams
from ebms_sim.profiles import generate_speed_profile, generate_gradient_profile
from ebms_sim.trip import simulate_trip
from ebms_sim.optimizer import sweep_speed_wh_per_km, optimize_trip_speeds_dp, RouteSegment
from ebms_sim.charging import simulate_charge, temperature_charge_multiplier, max_charge_current_a

print("ebms_sim loaded successfully.")

## 1. 10-Minute Trip Simulation
Simulates an Indian e-rickshaw carrying 3 passengers on a mixed route with stops, acceleration, hill climb, and descent.

In [ ]:
preset = PRESETS["lead_acid_48V_100Ah"]
duration_s = 600

speed_profile = generate_speed_profile(
    duration_s=duration_s,
    speed_cap_kmh=24.0,
    stop_interval_s=60.0,
    stop_duration_s=10.0,
    aggressiveness=1.0,
    seed=42,
)
gradient_profile = generate_gradient_profile(
    length_steps=duration_s,
    route_type="mixed",
    seed=42,
)

trip = simulate_trip(
    vehicle_params=preset["vehicle"],
    battery_params=preset["battery"],
    thermal_params=preset["thermal"],
    speed_profile_ms=speed_profile,
    gradient_profile=gradient_profile,
    payload_mass_kg=195.0,  # 3 passengers @ 65 kg
    t_ambient_c=32.0,
    headwind_ms=1.5,
    start_soc=0.85,
    dt_s=1.0,
)

summary = trip.summary
print(f"Distance:          {summary.distance_km:.2f} km")
print(f"Total Energy:      {summary.total_energy_wh:.1f} Wh")
print(f"Consumption:       {summary.wh_per_km:.1f} Wh/km")
print(f"Remaining Range:   {summary.range_remaining_km:.1f} km")
print(f"Final Pack Temp:   {summary.final_temp_c:.1f} °C")

In [ ]:
df = trip.time_series
time_min = df["time_s"] / 60.0

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(time_min, df["speed_kmh"], color="tab:blue", label="Speed (km/h)")
axes[0].set_ylabel("Speed [km/h]")
ax0_r = axes[0].twinx()
ax0_r.plot(time_min, df["elevation_m"], color="tab:brown", linestyle=":", label="Elevation (m)")
ax0_r.set_ylabel("Elevation [m]")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(time_min, df["battery_power_w"], color="tab:red", label="Battery Power (W)")
axes[1].axhline(0, color="black", linestyle="--", lw=0.8)
axes[1].set_ylabel("Power [W]")
axes[1].grid(True, linestyle="--", alpha=0.5)

axes[2].plot(time_min, df["soc"] * 100.0, color="tab:green", label="SOC (%)")
axes[2].set_ylabel("SOC [%]")
axes[2].set_xlabel("Time [minutes]")
ax2_r = axes[2].twinx()
ax2_r.plot(time_min, df["temp_c"], color="tab:red", linestyle="-.", label="Temp (°C)")
ax2_r.set_ylabel("Pack Temp [°C]")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 2. Wh/km vs Speed U-Curve Optimization
Evaluates specific energy consumption across speeds (10-25 km/h) to reveal optimal cruising speed.

In [ ]:
speeds = np.linspace(10.0, 25.0, 31)
spds, wh_flat, best_flat, min_flat = sweep_speed_wh_per_km(
    vehicle_params=preset["vehicle"],
    battery_params=preset["battery"],
    payload_mass_kg=130.0,
    grade_fraction=0.0,
    headwind_ms=0.0,
    t_ambient_c=25.0,
    speeds_kmh=speeds,
)

plt.figure(figsize=(8, 4.5))
plt.plot(spds, wh_flat, color="tab:blue", lw=2.5, label="Flat Road")
plt.scatter([best_flat], [min_flat], color="tab:red", s=80, zorder=5, label=f"Optimum: {best_flat:.1f} km/h ({min_flat:.1f} Wh/km)")
plt.title("Wh/km vs Speed U-Curve")
plt.xlabel("Cruising Speed [km/h]")
plt.ylabel("Consumption [Wh/km]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

## 3. Smart Charging: Thermal Bell Curve & CC-CV Taper

In [ ]:
temps = np.linspace(-5.0, 50.0, 200)
mults = [temperature_charge_multiplier(t) for t in temps]

plt.figure(figsize=(8, 4))
plt.plot(temps, mults, color="tab:red", lw=2.5)
plt.title("Charge Current Limit Thermal Derating Multiplier (Bell-Shaped)")
plt.xlabel("Pack Temperature [°C]")
plt.ylabel("Current Multiplier [0..1]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()